<a href="https://colab.research.google.com/github/fahadahmad84-design/Quantum-Computing-Labs/blob/main/notebooks/Week_07_QC_M34099_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 7 Laboratory: Grover Search and Shor's Algorithm


[![View in GitHub](https://img.shields.io/badge/View%20in-GitHub-black?logo=github)](https://github.com/fahadahmad84-design/Quantum-Computing-Labs/blob/main/notebooks/Week_07_QC_M34099_Student.ipynb)

**M34099 Quantum Computing: Level 7 (2026/27)**

This laboratory contains three basic tasks:

1. Calculate how the Grover diffuser changes four amplitudes;
2. Build and run a two-qubit Grover search; and
3. Find the period and factors for the small Shor example $N=15$ and $a=2$.

> **Saving your work:** Select **File > Save a copy in Drive** before editing. Never place an API key or CRN in a visible notebook cell.

## Set up Google Colab

Run the installation cell once. If Colab requests a restart, select **Runtime > Restart session**, then continue from the imports cell.

In [ ]:
# Install Qiskit, the Aer simulator, IBM Runtime, and drawing support.
%pip install -q "qiskit[visualization]" qiskit-aer qiskit-ibm-runtime pylatexenc

In [ ]:
# Import the greatest common divisor function.
from math import gcd
# Import NumPy for simple amplitude calculations.
import numpy as np
# Import the Qiskit circuit and transpiler tools.
from qiskit import QuantumCircuit, transpile
# Import the local ideal simulator.
from qiskit_aer import AerSimulator
# Import the histogram plotting function.
from qiskit.visualization import plot_histogram

# Set the number of repeated measurements.
SHOTS = 1024
# Set a fixed seed for repeatable simulator results.
SEED = 34099
# Create the ideal simulator.
simulator = AerSimulator()
# Confirm that the setup is ready.
print("Setup complete.")

## Task 1: Reflect amplitudes about their mean

For four states, the marked state $|11\rangle$ has its sign changed by the oracle:

$$\frac{1}{2}[1,1,1,-1]^T.$$

The diffuser changes each amplitude using

$$a_i' = 2\bar{a}-a_i,$$

where $\bar{a}$ is the mean amplitude.

In [ ]:
# Store the four amplitudes after the oracle marks state 11.
amplitudes = np.array([0.5, 0.5, 0.5, -0.5])
# Calculate the mean of the four amplitudes.
mean_amplitude = np.mean(amplitudes)
# Reflect every amplitude about the mean.
reflected_amplitudes = 2 * mean_amplitude - amplitudes
# Convert the final amplitudes into probabilities.
probabilities = np.abs(reflected_amplitudes) ** 2
# Display the mean amplitude.
print("Mean amplitude:", mean_amplitude)
# Display the amplitudes after the diffuser.
print("Amplitudes after diffuser:", reflected_amplitudes)
# Display the final probabilities.
print("Probabilities:", probabilities)

### Result

The mean amplitude is $1/4$. After reflection, the amplitudes become `[0, 0, 0, 1]`. The marked state $|11\rangle$ therefore has ideal probability 1.

## Task 2: Build a two-qubit Grover search

The oracle marks $|11\rangle$ by changing its phase. The diffuser then amplifies its amplitude. The code runs zero, one, two and three Grover iterations so that the effect of over-rotation is visible.

In [ ]:
# Create the phase oracle that marks state 11.
def make_oracle():
    # Create a two-qubit circuit named Oracle.
    circuit = QuantumCircuit(2, name="Oracle")
    # Change the phase of state 11 with a controlled-Z gate.
    circuit.cz(0, 1)
    # Return the completed oracle.
    return circuit

# Create the two-qubit Grover diffuser.
def make_diffuser():
    # Create a two-qubit circuit named Diffuser.
    circuit = QuantumCircuit(2, name="Diffuser")
    # Move from the computational basis to the Hadamard basis.
    circuit.h([0, 1])
    # Change both zero states into one states.
    circuit.x([0, 1])
    # Change the phase of state 11.
    circuit.cz(0, 1)
    # Undo the X gates.
    circuit.x([0, 1])
    # Undo the H gates.
    circuit.h([0, 1])
    # Return the completed diffuser.
    return circuit

# Create a complete Grover circuit.
def make_grover_circuit(iterations=1):
    # Create two qubits and two classical bits.
    circuit = QuantumCircuit(2, 2)
    # Create an equal superposition of 00, 01, 10, and 11.
    circuit.h([0, 1])
    # Repeat the Grover oracle and diffuser when requested.
    for _ in range(iterations):
        # Mark state 11 by changing its phase.
        circuit.compose(make_oracle(), inplace=True)
        # Amplify the marked-state amplitude.
        circuit.compose(make_diffuser(), inplace=True)
    # Measure both qubits.
    circuit.measure([0, 1], [0, 1])
    # Return the completed circuit.
    return circuit

# Build the circuit with one Grover iteration.
grover_one_iteration = make_grover_circuit(iterations=1)
# Draw the circuit.
display(grover_one_iteration.draw("mpl"))

In [ ]:
# Create an empty dictionary for simulator counts.
simulator_grover_counts = {}
# Create an empty dictionary for marked-state probabilities.
success_probabilities = {}

# Test zero, one, two, and three Grover iterations.
for iterations in range(4):
    # Build the circuit for the selected iteration count.
    circuit = make_grover_circuit(iterations=iterations)
    # Convert the circuit for the simulator.
    compiled = transpile(circuit, simulator, seed_transpiler=SEED)
    # Run the compiled circuit on the ideal simulator.
    job = simulator.run(compiled, shots=SHOTS, seed_simulator=SEED)
    # Collect the measurement counts.
    counts = job.result().get_counts()
    # Store the counts with a clear label.
    simulator_grover_counts[f"{iterations} iteration(s)"] = counts
    # Calculate the probability of measuring the marked state 11.
    success_probability = counts.get("11", 0) / SHOTS
    # Store the marked-state probability.
    success_probabilities[iterations] = success_probability
    # Display the result for this iteration count.
    print(iterations, "iteration(s):", success_probability)

# Compare the four measurement distributions.
display(plot_histogram(
    list(simulator_grover_counts.values()),
    legend=list(simulator_grover_counts.keys()),
    title="Grover search: effect of iteration count",
))

### Simulator result

With zero iterations, every state has probability close to $1/4$. One Grover iteration ideally returns `11` with probability 1. Additional iterations rotate probability away from the marked state, showing why more iterations are not always better.

## Task 3: Find the period and factors for $N=15$

Shor's algorithm reduces factoring to period finding. For this small example, the code displays the values of $2^x \bmod 15$ and identifies the first positive exponent that returns the value 1.

This transparent calculation illustrates the period and classical factor-recovery stages. It does not implement full quantum phase estimation.

In [ ]:
# Store the number that will be factored.
N = 15
# Store a value that is coprime with N.
a = 2
# Create an empty list for the modular values.
modular_values = []

# Calculate the first five values of 2 raised to x modulo 15.
for exponent in range(5):
    # Calculate the modular value for this exponent.
    remainder = pow(a, exponent, N)
    # Store the exponent and its remainder.
    modular_values.append((exponent, remainder))

# Display the modular sequence.
print("x and 2^x mod 15:", modular_values)

# Find the smallest positive exponent that returns remainder one.
period = next(exponent for exponent in range(1, N) if pow(a, exponent, N) == 1)
# Calculate a raised to half of the period.
half_period_value = pow(a, period // 2)
# Calculate the first possible non-trivial factor.
factor_one = gcd(half_period_value - 1, N)
# Calculate the second possible non-trivial factor.
factor_two = gcd(half_period_value + 1, N)
# Display the period.
print("Period:", period)
# Display the two recovered factors.
print("Factors:", factor_one, "and", factor_two)

### Result

The sequence is `1, 2, 4, 8, 1`, so the period is $r=4$. The classical recovery step gives

$$\gcd(2^{r/2}-1,15)=3$$

and

$$\gcd(2^{r/2}+1,15)=5.$$

Therefore, $15=3\times5$. This small example demonstrates the workflow but does not show that present hardware can factor practical RSA keys.

## Optional: Run the one-iteration Grover circuit on IBM quantum hardware

Run this section only after your tutor confirms that IBM Quantum Classroom access is ready. The same one-iteration Grover circuit is used so the ideal simulator and hardware results can be compared directly.

- Keep `RUN_ON_REAL_HARDWARE = False` until access is ready.
- Enter your API key and classroom CRN only through the hidden prompts.
- Never save credentials in the notebook, Google Drive, Moodle or GitHub.

In [ ]:
# Keep real-hardware execution disabled until access is ready.
RUN_ON_REAL_HARDWARE = False
# Display the current hardware setting.
print("Real-hardware execution enabled:", RUN_ON_REAL_HARDWARE)

In [ ]:
# Continue only when real-hardware execution is enabled.
if RUN_ON_REAL_HARDWARE:
    # Import secure hidden-input support.
    from getpass import getpass
    # Import the IBM Quantum Runtime connection.
    from qiskit_ibm_runtime import QiskitRuntimeService
    # Import the IBM Runtime Sampler interface.
    from qiskit_ibm_runtime import SamplerV2
    # Import the device-aware circuit pass manager.
    from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

    # Connect only once during the current Colab session.
    if "ibm_service" not in globals():
        # Request the API key without displaying it.
        ibm_api_key = getpass("Enter your personal IBM Cloud API key: ")
        # Request the classroom CRN without displaying it.
        classroom_crn = getpass("Enter the classroom instance CRN: ")
        # Connect to the classroom instance.
        ibm_service = QiskitRuntimeService(
            channel="ibm_quantum_platform",
            token=ibm_api_key,
            instance=classroom_crn,
        )
        # Remove the plaintext credential variables.
        del ibm_api_key, classroom_crn

    # Select an operational quantum computer with a short queue.
    backend = ibm_service.least_busy(operational=True, simulator=False)
    # Display the selected device.
    print("Selected backend:", backend.name)
    # Create a pass manager for the selected device.
    pass_manager = generate_preset_pass_manager(backend=backend, optimization_level=1)
    # Convert the one-iteration Grover circuit for the real device.
    hardware_circuit = pass_manager.run(grover_one_iteration)
    # Create the Runtime Sampler.
    sampler = SamplerV2(mode=backend)
    # Submit the circuit as one hardware job.
    hardware_job = sampler.run([hardware_circuit], shots=SHOTS)
    # Display the job identifier.
    print("Job ID:", hardware_job.job_id())
    # Wait for the job and collect the result.
    hardware_result = hardware_job.result()
    # Extract ordinary measurement counts from the result.
    hardware_counts = hardware_result[0].data.c.get_counts()
    # Obtain the matching ideal simulator counts.
    ideal_counts = simulator_grover_counts["1 iteration(s)"]
    # Display the hardware counts.
    print("Hardware results:", hardware_counts)
    # Compare ideal and hardware measurement distributions.
    display(plot_histogram(
        [ideal_counts, hardware_counts],
        legend=["Ideal simulator", f"IBM {backend.name}"],
        title="Grover search: simulator and hardware comparison",
    ))
# Skip the hardware job when the safety switch is off.
else:
    # Confirm that no hardware job was submitted.
    print("No hardware job submitted.")

### Hardware result

The ideal simulator should return only `11` after one Grover iteration. Real hardware may also return other bit strings because physical qubits and gates are noisy. The marked result `11` should remain the most frequent outcome when the circuit runs successfully.

## References

- IBM Quantum Learning, *Grover's algorithm*: https://quantum.cloud.ibm.com/learning/en/courses/fundamentals-of-quantum-algorithms/grover-algorithm
- IBM Quantum Learning, *Shor's algorithm*: https://quantum.cloud.ibm.com/learning/en/courses/fundamentals-of-quantum-algorithms/phase-estimation-and-factoring
- IBM Quantum Documentation, *Run jobs with Qiskit Runtime*: https://quantum.cloud.ibm.com/docs/guides/run-jobs
- Qiskit Aer documentation: https://qiskit.github.io/qiskit-aer/

Accessed 9 November 2026.